# Benchmark методов поиска оптимального набора факторов

Ноутбук сравнивает несколько стратегий отбора факторов на едином контуре **CatBoost + Gini** для бинарной классификации.

Принципы эксперимента:
- `TRAIN` используется для обучения;
- `VALID` — для выбора состава факторов и остановки по инкрементальному приросту Gini;
- `OOS` не участвует в поиске и используется только для финальной независимой оценки найденного набора;
- все методы получают один и тот же исходный пул факторов и одинаковые параметры модели;
- сохраняются время, число обучений, траектория поиска, итоговый состав факторов, стабильность между повторами и drift-диагностика.

Методы: `greedy`, `beam`, `random`, `TPE`, `Gaussian Process`, `CMA-ES`.

## Изменения протокола v2

Эта версия предназначена для честного сравнения методов поиска состава факторов.

Ключевые правила:

- **OOS не участвует** в отборе факторов, preprocessing, stopping и выборе гиперпараметров.
- Финальный набор определяется только по TRAIN/VALID.
- Сохраняется лучший найденный набор, а не последний набор перед остановкой.
- Ограничение `min_features_before_stop` относится к фактическому числу факторов.
- Все `model.fit()` учитываются в вычислительном бюджете.
- Отдельно измеряются время поиска, финального обучения и полное время.
- Baseline на всех факторах должен запускаться на тех же seed, что и методы поиска.
- Автоматически категориальными считаются только `object`, `category` и `bool`; числовые low-cardinality признаки не переводятся в категории без явного разрешения.
- Drift-диагностика должна считаться на исходных данных до технических преобразований CatBoost.
- Для сравнения количества факторов сохраняется frontier: `N факторов → лучший найденный VALID Gini`.
- OOS используется только для однократной финальной оценки выбранного решения каждого независимого запуска.

> Важно: VALID всё ещё используется как eval-set CatBoost и как критерий feature search. Для строгого исследовательского режима рекомендуется отдельный внутренний eval/CV внутри TRAIN; OOS при этом остаётся полностью закрытым до финала.


In [ ]:
# Если библиотек нет в окружении, раскомментируйте строку ниже.
# %pip install pandas numpy scikit-learn catboost optuna cmaes pyarrow openpyxl

from __future__ import annotations

import json
import math
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

try:
    import optuna
except ImportError:
    optuna = None

## 1. Настройки

Укажите пути к `TRAIN`, `VALID`, `OOS`, название target и при необходимости служебные/категориальные поля. `delta_gini` задаёт минимальный инкрементальный прирост, а `plateau_patience` — сколько последовательных слабых шагов допускается до остановки.

In [ ]:
@dataclass
class Config:
    train: str = ""
    valid: str = ""
    oos: str = ""
    target: str = "target"
    id_columns: List[str] = field(default_factory=list)
    exclude_columns: List[str] = field(default_factory=list)
    categorical_columns: List[str] = field(default_factory=list)
    auto_categorical: bool = True
    max_auto_cat_unique: int = 50

    # Обработка дат и неоднозначных типов.
    # Явно перечисленные datetime_columns имеют приоритет над автоопределением.
    datetime_columns: List[str] = field(default_factory=list)
    auto_datetime: bool = True
    datetime_parse_threshold: float = 0.95
    datetime_features: List[str] = field(
        default_factory=lambda: ["timestamp", "year", "month", "day", "dayofweek"]
    )

    # По умолчанию low-cardinality integer НЕ переводим в категорию автоматически.
    auto_integer_categorical: bool = False

    # Если object/string не является датой, он считается категориальным.
    # Сложные Python-объекты (list/dict/array и т. п.) останавливают запуск с понятной ошибкой.

    methods: List[str] = field(default_factory=lambda: ["greedy", "beam", "tpe", "gp", "cmaes", "random"])
    repeats: int = 3
    seeds: List[int] = field(default_factory=lambda: [42, 314, 2718])

    # Настройки поиска факторов
    start_features: List[str] = field(default_factory=list)
    max_features: Optional[int] = None
    delta_gini: float = 0.001
    plateau_patience: int = 3
    min_features_before_stop: int = 5
    additions_per_step: int = 1
    candidate_pool_size: int = 30
    beam_width: int = 5
    random_trials_per_step: int = 30
    optimizer_trials_per_step: int = 30
    optimizer_timeout_per_step: Optional[int] = None

    # Жёсткие ограничения бюджета. 0/None означает отсутствие ограничения.
    max_model_fits: Optional[int] = None
    max_wall_seconds: Optional[int] = None

    # Параметры CatBoost фиксируются на время поиска факторов для честного сравнения.
    catboost_params: Dict[str, Any] = field(default_factory=lambda: {
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": -1,
        "random_seed": 42,
    })
    early_stopping_rounds: int = 75

    output_dir: str = "feature_benchmark"
    save_models: bool = False

# ---------------- НАСТРОЙКИ КОНКРЕТНОГО ЗАПУСКА ----------------
cfg = Config(
    train="train.parquet",
    valid="valid.parquet",
    oos="oos.parquet",
    target="target",

    # При необходимости перечислите ID, служебные и заведомо leakage-поля.
    id_columns=[],
    exclude_columns=[],

    # Можно задать вручную. При auto_categorical=True часть категорий будет определена автоматически.
    categorical_columns=[],
    auto_categorical=True,
    auto_integer_categorical=False,

    # Если даты известны заранее, лучше перечислить их здесь.
    # Иначе auto_datetime=True попытается определить строковые даты по TRAIN.
    datetime_columns=[],
    auto_datetime=True,

    methods=["greedy", "beam", "random", "tpe", "gp", "cmaes"],
    repeats=3,
    seeds=[42, 314, 2718],

    delta_gini=0.001,
    plateau_patience=3,
    min_features_before_stop=5,
    max_features=None,

    # 2+ позволяет проверять совместное добавление факторов, а не только одиночный прирост.
    additions_per_step=2,
    candidate_pool_size=30,
    beam_width=5,
    random_trials_per_step=30,
    optimizer_trials_per_step=30,

    # None = без жёсткого ограничения.
    max_model_fits=None,
    max_wall_seconds=None,

    output_dir="feature_benchmark",
)

cfg

In [ ]:
def validate_experiment_protocol(cfg):
    """Базовые проверки конфигурации до запуска дорогого benchmark."""
    if cfg.repeats != len(cfg.seeds):
        raise ValueError(
            f"repeats={cfg.repeats}, но задано {len(cfg.seeds)} seed. "
            "Для воспроизводимого сравнения эти значения должны совпадать."
        )
    if cfg.additions_per_step < 1:
        raise ValueError("additions_per_step должен быть >= 1.")
    if cfg.delta_gini < 0:
        raise ValueError("delta_gini не может быть отрицательным.")
    if cfg.plateau_patience < 1:
        raise ValueError("plateau_patience должен быть >= 1.")
    if cfg.max_features is not None and cfg.max_features < cfg.min_features_before_stop:
        raise ValueError(
            "max_features не может быть меньше min_features_before_stop."
        )
    if not cfg.target:
        raise ValueError("Не задан target.")
    print("Проверка конфигурации пройдена.")

validate_experiment_protocol(cfg)


## 2. Загрузка, аудит типов и минимальный preprocessing

Перед первым обучением ноутбук строит `type_audit.csv`.

### Обработка дат в v4

Автоопределение дат теперь двухступенчатое и выполняется **только по TRAIN**:

1. Быстрый regex-фильтр проверяет небольшой sample строк и отбрасывает обычные текстовые/категориальные факторы без попытки парсинга.
2. Только колонки, действительно похожие на даты, проверяются векторным `pd.to_datetime()` с явными форматами / `format="mixed"`.

Это устраняет поток `Could not infer format...`, не запускает дорогой `dateutil` для каждой строковой колонки и заметно ускоряет preprocessing на широких витринах.

Настоящий `datetime64` определяется напрямую. После распознавания исходная дата заменяется на числовые компоненты (`timestamp`, `year`, `month`, `day`, `dayofweek`). VALID и OOS получают ту же TRAIN-схему и не участвуют в выборе преобразования.

Сложные объектные поля (`list`, `dict`, массивы и т. п.) останавливают запуск до первого CatBoost fit с понятным аудитом типов.


In [ ]:
def load_table(path: str) -> pd.DataFrame:
    """
    Загружает таблицу из одиночного файла или директории с Parquet-part'ами.
    """
    p = Path(path).expanduser()
    if not p.exists():
        raise FileNotFoundError(f"Путь не найден: {p}")

    if p.is_dir():
        try:
            return pd.read_parquet(p)
        except Exception as directory_error:
            parquet_files = sorted(
                f for f in p.rglob("*")
                if f.is_file() and f.suffix.lower() in {".parquet", ".pq"}
            )
            if not parquet_files:
                raise ValueError(
                    f"В директории {p} не найдено Parquet-файлов."
                ) from directory_error
            return pd.concat(
                [pd.read_parquet(f) for f in parquet_files],
                ignore_index=True,
            )

    ext = p.suffix.lower()
    if ext in {".parquet", ".pq"}:
        return pd.read_parquet(p)
    if ext in {".csv", ".txt"}:
        return pd.read_csv(p)
    if ext in {".pkl", ".pickle"}:
        return pd.read_pickle(p)
    if ext in {".xlsx", ".xls"}:
        return pd.read_excel(p)
    if ext == ".feather":
        return pd.read_feather(p)

    raise ValueError(f"Неподдерживаемый формат файла: {p.suffix}")


def validate_binary_target(y: pd.Series, name: str) -> pd.Series:
    """Проверяет бинарный target и при необходимости детерминированно переводит его в 0/1."""
    if y.isna().any():
        raise ValueError(f"{name}: в target есть пропуски")
    vals = list(pd.unique(y))
    if len(vals) != 2:
        raise ValueError(f"{name}: нужен бинарный target, получены значения {vals[:10]}")
    if set(vals) == {0, 1} or set(vals) == {False, True}:
        return y.astype(int)
    mapping = {v: i for i, v in enumerate(sorted(vals, key=lambda x: str(x)))}
    warnings.warn(f"{name}: target преобразован в 0/1 по правилу {mapping}")
    return y.map(mapping).astype(int)


def _dtype_name(s: pd.Series) -> str:
    return str(s.dtype)


def _has_complex_python_objects(s: pd.Series, sample_size: int = 1000) -> bool:
    """
    Ищет значения, которые CatBoost не должен получать напрямую:
    list/dict/set/tuple/ndarray и другие вложенные структуры.
    """
    x = s.dropna().head(sample_size)
    complex_types = (list, dict, set, tuple, np.ndarray)
    return any(isinstance(v, complex_types) for v in x)


def _looks_like_datetime_strings(
    s: pd.Series,
    sample_size: int = 500,
    min_pattern_rate: float = 0.80,
) -> Tuple[bool, float]:
    """
    Дешёвый предварительный фильтр для строковых дат.

    Сначала проверяем только форму строк и НЕ запускаем dateutil.
    В парсинг попадают лишь колонки, где большинство непустых значений
    похоже на дату/дату-время: YYYY-MM-DD, YYYY/MM/DD, DD.MM.YYYY,
    ISO datetime с T, дата со временем и т. п.

    Решение принимается только по TRAIN.
    """
    x = s.dropna()
    if x.empty:
        return False, 0.0

    if len(x) > sample_size:
        x = x.sample(sample_size, random_state=42)

    x = x.astype("string").str.strip()

    patterns = [
        r"^\d{4}[-/]\d{1,2}[-/]\d{1,2}(?:[ T].*)?$",
        r"^\d{1,2}[./-]\d{1,2}[./-]\d{4}(?:[ T].*)?$",
        r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}",
        r"^\d{4}\d{2}\d{2}$",
    ]

    mask = False
    for pattern in patterns:
        current = x.str.match(pattern, na=False)
        mask = current if isinstance(mask, bool) else (mask | current)

    rate = float(mask.mean())
    return rate >= min_pattern_rate, rate


def _datetime_parse_rate(
    s: pd.Series,
    sample_size: int = 500,
) -> Tuple[float, Optional[str]]:
    """
    Подтверждает дату только после дешёвого regex-фильтра.

    Сначала пробуем несколько распространённых форматов в векторном режиме.
    Только если они не подходят, используем format='mixed' (Pandas >= 2),
    также без предупреждений о невозможности вывести единый формат.

    Возвращает:
      parse_rate — долю успешно распарсенных значений;
      format_hint — найденный формат либо 'mixed'.
    """
    x = s.dropna()
    if x.empty:
        return 0.0, None

    if len(x) > sample_size:
        x = x.sample(sample_size, random_state=42)

    x = x.astype("string").str.strip()

    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d.%m.%Y",
        "%d-%m-%Y",
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%dT%H:%M:%S.%f",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S.%f",
        "%Y%m%d",
    ]

    best_rate = 0.0
    best_format = None

    for fmt in formats:
        parsed = pd.to_datetime(x, format=fmt, errors="coerce")
        rate = float(parsed.notna().mean())
        if rate > best_rate:
            best_rate = rate
            best_format = fmt
        if rate >= 0.995:
            return rate, fmt

    # Для смешанных, но явно похожих на даты значений.
    try:
        parsed = pd.to_datetime(x, format="mixed", errors="coerce")
        mixed_rate = float(parsed.notna().mean())
        if mixed_rate > best_rate:
            return mixed_rate, "mixed"
    except (TypeError, ValueError):
        # Совместимость со старыми версиями pandas:
        # не запускаем медленный неявный dateutil-парсинг.
        pass

    return best_rate, best_format


def detect_datetime_columns(
    train: pd.DataFrame,
    features: Sequence[str],
    cfg: Config,
) -> Tuple[List[str], Dict[str, str]]:
    """
    Определяет даты только по TRAIN.
    Явно заданные datetime_columns имеют приоритет.
    Автоопределение применяется к настоящим datetime dtype и строковым/object полям.
    Чистые числовые признаки автоматически датами не считаются.
    """
    detected = []
    reason = {}
    explicit = set(cfg.datetime_columns)

    for c in features:
        s = train[c]

        if c in explicit:
            detected.append(c)
            reason[c] = "DATETIME_EXPLICIT"
            continue

        if pd.api.types.is_datetime64_any_dtype(s):
            detected.append(c)
            reason[c] = "DATETIME_DTYPE"
            continue

        if not cfg.auto_datetime:
            continue

        if (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_string_dtype(s)
        ):
            looks_like_date, pattern_rate = _looks_like_datetime_strings(s)

            # Обычные текстовые/категориальные поля дальше не парсим вообще.
            if not looks_like_date:
                continue

            rate, format_hint = _datetime_parse_rate(s)
            if rate >= cfg.datetime_parse_threshold:
                detected.append(c)
                reason[c] = (
                    f"DATETIME_AUTO_parse={rate:.3f};"
                    f"pattern={pattern_rate:.3f};format={format_hint}"
                )

    return detected, reason


def expand_datetime_columns(
    train: pd.DataFrame,
    valid: pd.DataFrame,
    oos: pd.DataFrame,
    datetime_cols: Sequence[str],
    cfg: Config,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, List[str]]]:
    """
    Преобразует дату в числовые признаки.
    Схема преобразования определяется заранее и одинакова для TRAIN/VALID/OOS.
    OOS не участвует в решении, какие даты или компоненты использовать.
    """
    train = train.copy()
    valid = valid.copy()
    oos = oos.copy()
    generated = {}

    for c in datetime_cols:
        new_cols = []

        for df_name, df in [("TRAIN", train), ("VALID", valid), ("OOS", oos)]:
            # Явный/mixed формат не даёт Pandas перебирать dateutil
            # для каждой строки и не создаёт поток UserWarning.
            try:
                parsed = pd.to_datetime(df[c], format="mixed", errors="coerce")
            except (TypeError, ValueError):
                # Для старых Pandas используем ISO8601; явно заданные нестандартные
                # даты при необходимости лучше перечислить в datetime_columns
                # и предварительно привести к единому формату.
                parsed = pd.to_datetime(df[c], format="ISO8601", errors="coerce")

            # Если явно/автоматически признанная дата вообще не парсится на TRAIN,
            # это ошибка схемы, а не повод молча менять решение по VALID/OOS.
            if df_name == "TRAIN" and parsed.notna().sum() == 0 and df[c].notna().sum() > 0:
                raise ValueError(
                    f"Колонка '{c}' определена как дата, но значения TRAIN не удалось распарсить."
                )

            if "timestamp" in cfg.datetime_features:
                # Unix timestamp в секундах. NaT -> NaN.
                values = parsed.astype("int64", copy=False).astype("float64") / 1e9
                values[parsed.isna()] = np.nan
                df[f"{c}__timestamp"] = values

            if "year" in cfg.datetime_features:
                df[f"{c}__year"] = parsed.dt.year.astype("float64")
            if "month" in cfg.datetime_features:
                df[f"{c}__month"] = parsed.dt.month.astype("float64")
            if "day" in cfg.datetime_features:
                df[f"{c}__day"] = parsed.dt.day.astype("float64")
            if "dayofweek" in cfg.datetime_features:
                df[f"{c}__dayofweek"] = parsed.dt.dayofweek.astype("float64")
            if "quarter" in cfg.datetime_features:
                df[f"{c}__quarter"] = parsed.dt.quarter.astype("float64")

        for suffix in cfg.datetime_features:
            name = f"{c}__{suffix}"
            if name in train.columns:
                new_cols.append(name)

        generated[c] = new_cols

        # Исходная дата в CatBoost больше не передаётся.
        train.drop(columns=[c], inplace=True)
        valid.drop(columns=[c], inplace=True)
        oos.drop(columns=[c], inplace=True)

    return train, valid, oos, generated


def build_type_audit(
    train_raw: pd.DataFrame,
    valid_raw: pd.DataFrame,
    oos_raw: pd.DataFrame,
    candidates: Sequence[str],
    datetime_cols: Sequence[str],
    datetime_reason: Dict[str, str],
) -> pd.DataFrame:
    """Формирует понятный аудит типов до запуска первого CatBoost fit."""
    rows = []
    dt_set = set(datetime_cols)

    for c in candidates:
        action = "KEEP"
        note = ""

        if c in dt_set:
            action = "EXPAND_DATETIME"
            note = datetime_reason.get(c, "")
        elif _has_complex_python_objects(train_raw[c]):
            action = "ERROR_COMPLEX_OBJECT"
            note = "В колонке есть list/dict/array/tuple/set"
        elif pd.api.types.is_numeric_dtype(train_raw[c]):
            action = "NUMERIC"
        elif (
            pd.api.types.is_object_dtype(train_raw[c])
            or pd.api.types.is_string_dtype(train_raw[c])
            or isinstance(train_raw[c].dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(train_raw[c])
        ):
            action = "CATEGORICAL"
        else:
            action = "UNSUPPORTED_DTYPE"
            note = f"Необработанный dtype TRAIN: {train_raw[c].dtype}"

        rows.append({
            "feature": c,
            "dtype_train": _dtype_name(train_raw[c]),
            "dtype_valid": _dtype_name(valid_raw[c]),
            "dtype_oos": _dtype_name(oos_raw[c]),
            "action": action,
            "note": note,
        })

    return pd.DataFrame(rows)


def infer_categoricals(
    train: pd.DataFrame,
    cfg: Config,
    features: Sequence[str],
) -> List[str]:
    """
    Категориальность определяется по TRAIN.
    По умолчанию integer с малым числом значений НЕ считается категорией автоматически.
    """
    explicit = [c for c in cfg.categorical_columns if c in features]
    if not cfg.auto_categorical:
        return sorted(set(explicit))

    cats = set(explicit)
    for c in features:
        s = train[c]
        if (
            pd.api.types.is_object_dtype(s)
            or pd.api.types.is_string_dtype(s)
            or isinstance(s.dtype, pd.CategoricalDtype)
            or pd.api.types.is_bool_dtype(s)
        ):
            cats.add(c)
        elif (
            cfg.auto_integer_categorical
            and pd.api.types.is_integer_dtype(s)
            and s.nunique(dropna=True) <= cfg.max_auto_cat_unique
        ):
            cats.add(c)
    return sorted(cats)


def technical_filter(
    train: pd.DataFrame,
    valid: pd.DataFrame,
    oos: pd.DataFrame,
    cfg: Config,
):
    """
    Технический preprocessing без target-based отбора.

    Важно:
    - решения о типах и удалении факторов принимаются только по TRAIN;
    - VALID используется для обучения/выбора позже;
    - OOS здесь только преобразуется по уже принятой TRAIN-схеме;
    - OOS никогда не определяет, оставить или удалить фактор.
    """
    for name, df in [("TRAIN", train), ("VALID", valid), ("OOS", oos)]:
        if cfg.target not in df.columns:
            raise ValueError(f"{name}: отсутствует target '{cfg.target}'")

    # Схема признаков определяется по TRAIN.
    missing_valid = [c for c in train.columns if c not in valid.columns and c != cfg.target]
    missing_oos = [c for c in train.columns if c not in oos.columns and c != cfg.target]
    if missing_valid:
        raise ValueError(f"VALID: отсутствуют колонки TRAIN: {missing_valid[:20]}")
    if missing_oos:
        raise ValueError(f"OOS: отсутствуют колонки TRAIN: {missing_oos[:20]}")

    excluded = set([cfg.target] + cfg.id_columns + cfg.exclude_columns)
    candidates = [c for c in train.columns if c not in excluded]

    # Сначала аудитируем исходные типы.
    datetime_cols, datetime_reason = detect_datetime_columns(train, candidates, cfg)
    type_audit = build_type_audit(
        train, valid, oos, candidates, datetime_cols, datetime_reason
    )

    fatal = type_audit[type_audit["action"].isin(
        ["ERROR_COMPLEX_OBJECT", "UNSUPPORTED_DTYPE"]
    )]
    if not fatal.empty:
        display(fatal.head(50))
        raise TypeError(
            "Обнаружены факторы с неподдерживаемыми типами. "
            "Смотрите таблицу type_audit выше; такие поля нужно исключить "
            "или явно преобразовать до benchmark."
        )

    # Константы удаляем ТОЛЬКО по TRAIN.
    technical_rows = []
    base_keep = []
    for c in candidates:
        if train[c].nunique(dropna=False) <= 1:
            technical_rows.append({
                "feature": c,
                "status": "DROP_CONSTANT_TRAIN",
                "details": "Константа на TRAIN",
            })
        else:
            base_keep.append(c)
            technical_rows.append({
                "feature": c,
                "status": "KEEP_OR_TRANSFORM",
                "details": datetime_reason.get(c, ""),
            })

    datetime_cols = [c for c in datetime_cols if c in base_keep]

    # Даты раскладываем на числовые компоненты одинаково во всех трёх срезах.
    train_p, valid_p, oos_p, generated_dates = expand_datetime_columns(
        train, valid, oos, datetime_cols, cfg
    )

    # Итоговый список факторов: обычные + производные от дат.
    keep = [c for c in base_keep if c not in datetime_cols]
    for original in datetime_cols:
        keep.extend(generated_dates.get(original, []))

    # +/-inf заменяем на NaN только в числовых полях.
    for df in (train_p, valid_p, oos_p):
        numeric = [c for c in keep if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
        if numeric:
            df.loc[:, numeric] = df[numeric].replace([np.inf, -np.inf], np.nan)

    cats = infer_categoricals(train_p, cfg, keep)

    # CatBoost не принимает NaN как значение категориального признака.
    # Приведение схемы выполняется одинаково, но решение о cats принято только по TRAIN.
    for df in (train_p, valid_p, oos_p):
        for c in cats:
            df[c] = df[c].astype("string").fillna("__MISSING__")

    # Финальная защита: в числовые факторы не должны просочиться строки/datetime.
    bad_numeric = []
    for c in keep:
        if c in cats:
            continue
        if not pd.api.types.is_numeric_dtype(train_p[c]):
            bad_numeric.append((c, str(train_p[c].dtype)))

    if bad_numeric:
        raise TypeError(
            "После preprocessing остались нечисловые факторы, не объявленные категориальными: "
            + repr(bad_numeric[:20])
        )

    technical = pd.DataFrame(technical_rows)
    return train_p, valid_p, oos_p, keep, cats, technical, type_audit, generated_dates


## 3. Gini и единый контур обучения CatBoost

In [ ]:
def gini(y_true: Sequence[int], p: Sequence[float]) -> float:
    return 2.0 * roc_auc_score(y_true, p) - 1.0


class BudgetExceeded(RuntimeError):
    pass


class Evaluator:
    def __init__(
        self,
        train: pd.DataFrame,
        valid: pd.DataFrame,
        oos: pd.DataFrame,
        target: str,
        categorical: Sequence[str],
        cfg: Config,
        method: str,
        repeat: int,
        seed: int,
    ):
        self.train, self.valid, self.oos = train, valid, oos
        self.target = target
        self.categorical = set(categorical)
        self.cfg = cfg
        self.method, self.repeat, self.seed = method, repeat, seed
        self.started = time.perf_counter()
        self.fit_count = 0
        self.cache: Dict[Tuple[str, ...], Dict[str, Any]] = {}
        self.rows: List[Dict[str, Any]] = []

    def _check_budget(self):
        if self.cfg.max_model_fits and self.fit_count >= self.cfg.max_model_fits:
            raise BudgetExceeded("max_model_fits reached")
        if self.cfg.max_wall_seconds and (time.perf_counter() - self.started) >= self.cfg.max_wall_seconds:
            raise BudgetExceeded("max_wall_seconds reached")

    def evaluate(self, features: Sequence[str], stage: str, note: str = "") -> Dict[str, Any]:
        fs = tuple(sorted(set(features)))
        if not fs:
            raise ValueError("Cannot fit model with zero features")
        if fs in self.cache:
            result = dict(self.cache[fs])
            result["cached"] = True
            return result
        self._check_budget()

        cats = [c for c in fs if c in self.categorical]
        params = dict(self.cfg.catboost_params)
        params["random_seed"] = self.seed
        model = CatBoostClassifier(**params)
        t0 = time.perf_counter()
        model.fit(
            self.train[list(fs)], self.train[self.target],
            cat_features=cats,
            eval_set=(self.valid[list(fs)], self.valid[self.target]),
            early_stopping_rounds=self.cfg.early_stopping_rounds,
            verbose=False,
        )
        elapsed = time.perf_counter() - t0
        self.fit_count += 1
        pv = model.predict_proba(self.valid[list(fs)])[:, 1]
        gv = gini(self.valid[self.target], pv)
        result = {
            "features": list(fs), "n_features": len(fs), "valid_gini": gv,
            "fit_seconds": elapsed, "best_iteration": int(model.get_best_iteration()),
            "cached": False,
        }
        self.cache[fs] = dict(result)
        self.rows.append({
            "method": self.method, "repeat": self.repeat, "seed": self.seed,
            "fit_no": self.fit_count, "stage": stage, "note": note,
            "n_features": len(fs), "valid_gini": gv, "fit_seconds": elapsed,
            "elapsed_total": time.perf_counter() - self.started,
            "best_iteration": int(model.get_best_iteration()),
            "features": json.dumps(list(fs), ensure_ascii=False),
        })
        return result

    def final_oos(self, features: Sequence[str]) -> Dict[str, Any]:
        """OOS is called only after search has finished and never feeds search decisions."""
        fs = list(sorted(set(features)))
        cats = [c for c in fs if c in self.categorical]
        params = dict(self.cfg.catboost_params)
        params["random_seed"] = self.seed
        # Берём лучшую итерацию, найденную на TRAIN→VALID, затем переобучаемся на TRAIN+VALID.
        cached = self.cache.get(tuple(fs))
        best_iter = cached.get("best_iteration", params.get("iterations", 500) - 1) if cached else params.get("iterations", 500) - 1
        params["iterations"] = max(1, int(best_iter) + 1)
        full = pd.concat([self.train, self.valid], axis=0, ignore_index=True)
        model = CatBoostClassifier(**params)
        t0 = time.perf_counter()
        model.fit(full[fs], full[self.target], cat_features=cats, verbose=False)
        po = model.predict_proba(self.oos[fs])[:, 1]
        return {
            "oos_gini": gini(self.oos[self.target], po),
            "oos_fit_seconds": time.perf_counter() - t0,
            "final_iterations": params["iterations"],
        }

## 4. Вспомогательные функции и методы поиска

In [ ]:
def chunks_added(base: Sequence[str], remaining: Sequence[str], k: int) -> Iterable[List[str]]:
    # Для добавления одного фактора перебор полный. При k>1 полный перебор пар/групп намеренно
    # не используется из-за комбинаторного взрыва C(p,k): методы оценивают или сэмплируют группы кандидатов.
    if k != 1:
        raise ValueError("Exhaustive chunks_added supports additions_per_step=1 only")
    for f in remaining:
        yield list(base) + [f]


def plateau_update(history: List[float], delta: float, patience: int, min_steps: int) -> bool:
    if len(history) <= min_steps:
        return False
    if len(history) < patience + 1:
        return False
    recent = [history[i] - history[i - 1] for i in range(len(history) - patience, len(history))]
    return all(x < delta for x in recent)


def candidate_pool_by_single_addition(
    evaluator: Evaluator, selected: List[str], all_features: List[str], pool_size: int, stage: str
) -> List[Tuple[str, float]]:
    """Model-based marginal screen conditional on the CURRENT selected set, not univariate screening."""
    remaining = [f for f in all_features if f not in selected]
    scored = []
    for f in remaining:
        r = evaluator.evaluate(selected + [f], stage=stage, note=f"candidate={f}")
        scored.append((f, r["valid_gini"]))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[: min(pool_size, len(scored))]


# -----------------------------------------------------------------------------
# Search methods
# -----------------------------------------------------------------------------

def greedy_search(ev: Evaluator, features: List[str], cfg: Config) -> Dict[str, Any]:
    selected = [f for f in cfg.start_features if f in features]
    history: List[float] = []
    best = None
    if selected:
        best = ev.evaluate(selected, "greedy_start")
        history.append(best["valid_gini"])

    limit = cfg.max_features or len(features)
    try:
        while len(selected) < limit:
            remaining = [f for f in features if f not in selected]
            if not remaining:
                break
            candidates = []
            for f in remaining:
                r = ev.evaluate(selected + [f], "greedy_step", note=f"add={f}")
                candidates.append((r["valid_gini"], f, r))
            _, chosen, best = max(candidates, key=lambda x: x[0])
            selected.append(chosen)
            history.append(best["valid_gini"])
            if plateau_update(history, cfg.delta_gini, cfg.plateau_patience, cfg.min_features_before_stop):
                break
    except BudgetExceeded:
        pass
    if best is None:
        # Выбираем лучший одиночный фактор, если позволяет вычислительный бюджет.
        singles = [(ev.evaluate([f], "greedy_single")["valid_gini"], f) for f in features]
        _, f = max(singles)
        selected = [f]
        best = ev.cache[tuple(selected)]
    return {"features": selected, "valid_gini": best["valid_gini"], "history": history}


def beam_search(ev: Evaluator, features: List[str], cfg: Config) -> Dict[str, Any]:
    starts = [f for f in cfg.start_features if f in features]
    beam: List[Tuple[List[str], float]] = []
    history: List[float] = []
    limit = cfg.max_features or len(features)
    try:
        if starts:
            r = ev.evaluate(starts, "beam_start")
            beam = [(starts, r["valid_gini"])]
        else:
            singles = []
            for f in features:
                r = ev.evaluate([f], "beam_single", note=f"add={f}")
                singles.append(([f], r["valid_gini"]))
            singles.sort(key=lambda x: x[1], reverse=True)
            beam = singles[: cfg.beam_width]
        history.append(beam[0][1])

        while beam and len(beam[0][0]) < limit:
            expanded: Dict[Tuple[str, ...], float] = {}
            for subset, _ in beam:
                for f in features:
                    if f in subset:
                        continue
                    new = sorted(subset + [f])
                    key = tuple(new)
                    if key in expanded:
                        continue
                    r = ev.evaluate(new, "beam_expand", note=f"from={len(subset)}")
                    expanded[key] = r["valid_gini"]
            if not expanded:
                break
            ranked = sorted(expanded.items(), key=lambda kv: kv[1], reverse=True)[: cfg.beam_width]
            beam = [(list(k), v) for k, v in ranked]
            history.append(beam[0][1])
            if plateau_update(history, cfg.delta_gini, cfg.plateau_patience, cfg.min_features_before_stop):
                break
    except BudgetExceeded:
        pass
    if not beam:
        raise RuntimeError("Beam search produced no candidate")
    return {"features": beam[0][0], "valid_gini": beam[0][1], "history": history}


def random_search(ev: Evaluator, features: List[str], cfg: Config, rng: np.random.Generator) -> Dict[str, Any]:
    selected = [f for f in cfg.start_features if f in features]
    history = []
    best = None
    limit = cfg.max_features or len(features)
    try:
        while len(selected) < limit:
            remaining = [f for f in features if f not in selected]
            if not remaining:
                break
            k = min(cfg.additions_per_step, len(remaining))
            trials = []
            # При k=1 одиночные условные приросты всегда входят в пул кандидатов;
            # случайные альтернативы не дают поиску превратиться в чисто жадный.
            for t in range(cfg.random_trials_per_step):
                add = rng.choice(remaining, size=k, replace=False).tolist()
                r = ev.evaluate(selected + add, "random_step", note=f"trial={t}")
                trials.append((r["valid_gini"], add, r))
            _, add, best = max(trials, key=lambda x: x[0])
            selected.extend(add)
            history.append(best["valid_gini"])
            if plateau_update(history, cfg.delta_gini, cfg.plateau_patience, cfg.min_features_before_stop):
                break
    except BudgetExceeded:
        pass
    if best is None:
        f = rng.choice(features)
        selected = [str(f)]
        best = ev.evaluate(selected, "random_fallback")
    return {"features": selected, "valid_gini": best["valid_gini"], "history": history}


def _make_sampler(method: str, seed: int):
    if optuna is None:
        raise ImportError("Optuna is required for tpe/gp/cmaes methods")
    if method == "tpe":
        return optuna.samplers.TPESampler(seed=seed, multivariate=True)
    if method == "gp":
        if not hasattr(optuna.samplers, "GPSampler"):
            raise RuntimeError("Installed Optuna has no GPSampler; upgrade Optuna")
        return optuna.samplers.GPSampler(seed=seed)
    if method == "cmaes":
        return optuna.samplers.CmaEsSampler(seed=seed)
    raise ValueError(method)


def optuna_forward_search(ev: Evaluator, features: List[str], cfg: Config, method: str) -> Dict[str, Any]:
    """
    Optimizer-guided forward subset search.

    Each step first forms a conditional candidate pool by evaluating feature additions
    against the CURRENT subset. The optimizer then proposes a continuous score for each
    pool member; top-k scores define the proposed group. For k=1, TPE/GP/CMA-ES would
    otherwise be pointless over a single categorical variable; the continuous proposal
    representation keeps a common search interface and supports k>1 interactions.
    """
    selected = [f for f in cfg.start_features if f in features]
    history: List[float] = []
    best = None
    limit = cfg.max_features or len(features)
    step = 0
    try:
        while len(selected) < limit:
            remaining = [f for f in features if f not in selected]
            if not remaining:
                break
            k = min(cfg.additions_per_step, len(remaining))
            pool_scored = candidate_pool_by_single_addition(
                ev, selected, features, cfg.candidate_pool_size, stage=f"{method}_pool"
            )
            pool = [x[0] for x in pool_scored]
            if len(pool) <= k:
                add = pool
                best = ev.evaluate(selected + add, f"{method}_forced")
            elif k == 1:
                # При добавлении одного фактора условный пул уже исчерпывающе оценил
                # каждого кандидата, поэтому дополнительный сэмплинг оптимизатора не улучшит лучший вариант.
                # Поэтому оптимизатор имеет содержательный смысл здесь только при k>1.
                add = [pool[0]]
                best = ev.evaluate(selected + add, f"{method}_best_single")
            else:
                sampler = _make_sampler(method, ev.seed + step)
                study = optuna.create_study(direction="maximize", sampler=sampler)
                seen: Dict[Tuple[str, ...], float] = {}

                def objective(trial):
                    scores = [(trial.suggest_float(f"s{i}", 0.0, 1.0), f) for i, f in enumerate(pool)]
                    add_local = [f for _, f in sorted(scores, reverse=True)[:k]]
                    key = tuple(sorted(add_local))
                    if key not in seen:
                        r = ev.evaluate(selected + list(key), f"{method}_trial", note=f"step={step}")
                        seen[key] = r["valid_gini"]
                    return seen[key]

                study.optimize(
                    objective,
                    n_trials=cfg.optimizer_trials_per_step,
                    timeout=cfg.optimizer_timeout_per_step,
                    show_progress_bar=False,
                    catch=(BudgetExceeded,),
                )
                if not seen:
                    raise BudgetExceeded("No optimizer trial completed")
                add_key, _ = max(seen.items(), key=lambda kv: kv[1])
                add = list(add_key)
                best = ev.evaluate(selected + add, f"{method}_chosen")

            selected.extend([f for f in add if f not in selected])
            history.append(best["valid_gini"])
            step += 1
            if plateau_update(history, cfg.delta_gini, cfg.plateau_patience, cfg.min_features_before_stop):
                break
    except BudgetExceeded:
        pass

    if best is None:
        # Детерминированный запасной вариант: лучший одиночный фактор.
        pool = candidate_pool_by_single_addition(ev, [], features, 1, stage=f"{method}_fallback")
        selected = [pool[0][0]]
        best = ev.cache[tuple(selected)]
    return {"features": sorted(set(selected)), "valid_gini": best["valid_gini"], "history": history}

## 5. Диагностика drift и стабильности состава факторов

In [ ]:
def jaccard(a: Sequence[str], b: Sequence[str]) -> float:
    A, B = set(a), set(b)
    return len(A & B) / len(A | B) if (A | B) else 1.0


def population_stability_index(train_s: pd.Series, other_s: pd.Series, bins: int = 10) -> float:
    """Simple PSI diagnostic. Numeric uses TRAIN quantiles; categorical uses TRAIN levels."""
    eps = 1e-6
    try:
        if pd.api.types.is_numeric_dtype(train_s):
            clean = train_s.replace([np.inf, -np.inf], np.nan)
            qs = np.unique(clean.dropna().quantile(np.linspace(0, 1, bins + 1)).values)
            if len(qs) < 3:
                return 0.0
            qs[0], qs[-1] = -np.inf, np.inf
            a = pd.cut(clean, qs, include_lowest=True).value_counts(normalize=True, sort=False)
            b = pd.cut(other_s.replace([np.inf, -np.inf], np.nan), qs, include_lowest=True).value_counts(normalize=True, sort=False)
        else:
            a = train_s.astype("string").fillna("__MISSING__").value_counts(normalize=True)
            b = other_s.astype("string").fillna("__MISSING__").value_counts(normalize=True)
            idx = a.index.union(b.index)
            a, b = a.reindex(idx, fill_value=0), b.reindex(idx, fill_value=0)
        av, bv = np.clip(a.values.astype(float), eps, None), np.clip(b.values.astype(float), eps, None)
        return float(np.sum((bv - av) * np.log(bv / av)))
    except Exception:
        return float("nan")


def drift_report(train: pd.DataFrame, valid: pd.DataFrame, oos: pd.DataFrame, features: Sequence[str]) -> pd.DataFrame:
    rows = []
    for f in features:
        rows.append({
            "feature": f,
            "psi_train_valid": population_stability_index(train[f], valid[f]),
            "psi_train_oos": population_stability_index(train[f], oos[f]),
            "train_missing_rate": float(train[f].isna().mean()),
            "valid_missing_rate": float(valid[f].isna().mean()),
            "oos_missing_rate": float(oos[f].isna().mean()),
            "train_nunique": int(train[f].nunique(dropna=True)),
        })
    return pd.DataFrame(rows)

## 6. Запуск benchmark и формирование отчётов

In [ ]:
def run_one(method: str, repeat: int, seed: int, train, valid, oos, features, cats, cfg: Config):
    ev = Evaluator(train, valid, oos, cfg.target, cats, cfg, method, repeat, seed)
    rng = np.random.default_rng(seed)
    t0 = time.perf_counter()
    if method == "greedy":
        result = greedy_search(ev, features, cfg)
    elif method == "beam":
        result = beam_search(ev, features, cfg)
    elif method == "random":
        result = random_search(ev, features, cfg, rng)
    elif method in {"tpe", "gp", "cmaes"}:
        result = optuna_forward_search(ev, features, cfg, method)
    else:
        raise ValueError(f"Unknown method: {method}")
    search_seconds = time.perf_counter() - t0
    final = ev.final_oos(result["features"])
    summary = {
        "method": method, "repeat": repeat, "seed": seed,
        "search_seconds": search_seconds, "model_fits": ev.fit_count,
        "n_features": len(result["features"]), "valid_gini": result["valid_gini"],
        **final,
        "features": json.dumps(result["features"], ensure_ascii=False),
        "history": json.dumps(result["history"]),
    }
    return summary, pd.DataFrame(ev.rows)


def all_features_baseline(train, valid, oos, features, cats, cfg: Config) -> Dict[str, Any]:
    ev = Evaluator(train, valid, oos, cfg.target, cats, cfg, "all_features", 0, cfg.seeds[0])
    r = ev.evaluate(features, "baseline_all")
    o = ev.final_oos(features)
    return {
        "method": "all_features", "repeat": 0, "seed": cfg.seeds[0],
        "search_seconds": r["fit_seconds"], "model_fits": 1,
        "n_features": len(features), "valid_gini": r["valid_gini"], **o,
        "features": json.dumps(sorted(features), ensure_ascii=False), "history": "[]",
    }


def summarize_runs(runs: pd.DataFrame) -> pd.DataFrame:
    methods = runs[runs.method != "all_features"]
    rows = []
    for method, g in methods.groupby("method"):
        feature_sets = [json.loads(x) for x in g["features"]]
        jac = []
        for i in range(len(feature_sets)):
            for j in range(i + 1, len(feature_sets)):
                jac.append(jaccard(feature_sets[i], feature_sets[j]))
        rows.append({
            "method": method,
            "runs": len(g),
            "search_seconds_mean": g.search_seconds.mean(),
            "search_seconds_std": g.search_seconds.std(ddof=0),
            "model_fits_mean": g.model_fits.mean(),
            "n_features_mean": g.n_features.mean(),
            "n_features_std": g.n_features.std(ddof=0),
            "valid_gini_mean": g.valid_gini.mean(),
            "valid_gini_std": g.valid_gini.std(ddof=0),
            "oos_gini_mean": g.oos_gini.mean(),
            "oos_gini_std": g.oos_gini.std(ddof=0),
            "valid_oos_gap_mean": (g.valid_gini - g.oos_gini).mean(),
            "feature_jaccard_mean": float(np.mean(jac)) if jac else np.nan,
        })
    if not rows:
        return pd.DataFrame(columns=["method", "runs", "search_seconds_mean", "search_seconds_std", "model_fits_mean", "n_features_mean", "n_features_std", "valid_gini_mean", "valid_gini_std", "oos_gini_mean", "oos_gini_std", "valid_oos_gap_mean", "feature_jaccard_mean"])
    return pd.DataFrame(rows).sort_values("oos_gini_mean", ascending=False)


def write_outputs(out: Path, cfg: Config, runs: pd.DataFrame, trajectories: pd.DataFrame,
                  technical: pd.DataFrame, drift: pd.DataFrame):
    out.mkdir(parents=True, exist_ok=True)
    (out / "config_resolved.json").write_text(json.dumps(asdict(cfg), ensure_ascii=False, indent=2), encoding="utf-8")
    runs.to_csv(out / "runs.csv", index=False)
    trajectories.to_csv(out / "trajectories.csv", index=False)
    technical.to_csv(out / "technical_feature_audit.csv", index=False)
    drift.to_csv(out / "drift_report.csv", index=False)
    summary = summarize_runs(runs)
    summary.to_csv(out / "method_summary.csv", index=False)

    # Длинная таблица принадлежности факторов удобнее для сравнения, чем JSON-списки.
    membership = []
    for _, r in runs.iterrows():
        for f in json.loads(r["features"]):
            membership.append({"method": r.method, "repeat": r.repeat, "seed": r.seed, "feature": f})
    pd.DataFrame(membership).to_csv(out / "selected_features.csv", index=False)

    try:
        with pd.ExcelWriter(out / "benchmark_report.xlsx", engine="openpyxl") as xw:
            summary.to_excel(xw, "method_summary", index=False)
            runs.to_excel(xw, "runs", index=False)
            trajectories.to_excel(xw, "trajectories", index=False)
            technical.to_excel(xw, "technical_audit", index=False)
            drift.to_excel(xw, "drift", index=False)
            pd.DataFrame(membership).to_excel(xw, "selected_features", index=False)
    except Exception as e:
        warnings.warn(f"Excel report was not written: {e}")

In [ ]:
# Загружаем три независимых среза.
out = Path(cfg.output_dir)
out.mkdir(parents=True, exist_ok=True)

print("Загрузка датасетов...")
train = load_table(cfg.train)
valid = load_table(cfg.valid)
oos = load_table(cfg.oos)

train[cfg.target] = validate_binary_target(train[cfg.target], "TRAIN")
valid[cfg.target] = validate_binary_target(valid[cfg.target], "VALID")
oos[cfg.target] = validate_binary_target(oos[cfg.target], "OOS")

# Сохраняем исходные таблицы для диагностики типов и drift.
# Решения preprocessing принимаются только по TRAIN.
train_raw = train.copy()
valid_raw = valid.copy()
oos_raw = oos.copy()

train, valid, oos, features, cats, technical, type_audit, generated_dates = technical_filter(
    train, valid, oos, cfg
)
if not features:
    raise ValueError("После технической фильтрации не осталось доступных факторов")

print(f"Строки: TRAIN={len(train):,}, VALID={len(valid):,}, OOS={len(oos):,}")
print(f"Факторы после preprocessing: {len(features)}; категориальные: {len(cats)}")
print(f"Распознано исходных datetime-факторов: {len(generated_dates)}")

# Сохраняем аудит типов до первого CatBoost fit.
type_audit.to_csv(out / "type_audit.csv", index=False)
technical.to_csv(out / "technical_feature_audit.csv", index=False)

print("\nАудит типов (первые 50 строк):")
display(type_audit.head(50))

# Drift — только диагностика. Для обычных факторов используем исходные значения.
# Для производных datetime-факторов используем уже преобразованные числовые признаки.
print("Расчёт drift-диагностики...")
drift = drift_report(train, valid, oos, features)

# Baseline на полном наборе факторов нужен, чтобы понять, даёт ли feature selection практический выигрыш.
runs = [all_features_baseline(train, valid, oos, features, cats, cfg)]
trajectories = []

for method in cfg.methods:
    for repeat in range(cfg.repeats):
        seed = cfg.seeds[repeat] if repeat < len(cfg.seeds) else 42 + repeat * 997
        print(f"[{method}] повтор {repeat + 1}/{cfg.repeats}, seed={seed}")
        try:
            summary, traj = run_one(method, repeat, seed, train, valid, oos, features, cats, cfg)
            runs.append(summary)
            trajectories.append(traj)

            # Checkpoint: результаты уже завершённых запусков не потеряются при падении следующего метода.
            pd.DataFrame(runs).to_csv(out / "runs_checkpoint.csv", index=False)
            if trajectories:
                pd.concat(trajectories, ignore_index=True).to_csv(
                    out / "trajectories_checkpoint.csv", index=False
                )
        except Exception as e:
            err = {"method": method, "repeat": repeat, "seed": seed, "error": repr(e)}
            with (out / "errors.jsonl").open("a", encoding="utf-8") as fh:
                fh.write(json.dumps(err, ensure_ascii=False) + "\n")
            warnings.warn(f"Метод {method}, повтор {repeat}: ошибка — {e}")

runs_df = pd.DataFrame(runs)
traj_df = pd.concat(trajectories, ignore_index=True) if trajectories else pd.DataFrame()
write_outputs(out, cfg, runs_df, traj_df, technical, drift)

method_summary = summarize_runs(runs_df)
print("\n=== СРАВНЕНИЕ МЕТОДОВ ===")
display(method_summary)
print(f"Результаты сохранены в: {out.resolve()}")

## 7. Быстрый просмотр результатов

`method_summary` показывает средние время, число факторов, VALID/OOS Gini, разрыв VALID→OOS и стабильность состава факторов между повторами. Ниже можно отдельно посмотреть все запуски и траекторию поиска.

In [ ]:
display(runs_df.sort_values(["method", "repeat"]))

if not traj_df.empty:
    display(traj_df.head(50))

In [ ]:
def build_gini_frontier(trajectory_df, delta_gini=0.001, patience=3):
    """
    Строит frontier: для каждого количества факторов берётся лучший найденный VALID Gini.
    Plateau определяется по инкрементальному приросту лучшего Gini, а не по последнему
    случайно посещённому набору.
    """
    required = {"n_features", "valid_gini"}
    missing = required - set(trajectory_df.columns)
    if missing:
        raise ValueError(f"В trajectory отсутствуют поля: {sorted(missing)}")

    frontier = (
        trajectory_df[["n_features", "valid_gini"]]
        .dropna()
        .groupby("n_features", as_index=False)["valid_gini"]
        .max()
        .sort_values("n_features")
        .reset_index(drop=True)
    )

    frontier["best_so_far_gini"] = frontier["valid_gini"].cummax()
    frontier["incremental_gini"] = frontier["best_so_far_gini"].diff()

    best_n = None
    weak = 0
    previous = None

    for row in frontier.itertuples(index=False):
        current = row.best_so_far_gini
        if previous is None:
            previous = current
            best_n = row.n_features
            continue

        gain = current - previous
        if gain >= delta_gini:
            best_n = row.n_features
            weak = 0
            previous = current
        else:
            weak += 1
            if weak >= patience:
                break

    return frontier, best_n
